In [12]:
:dep enum-iterator = { version = "2" }

use enum_iterator::{all, cardinality, first, last, next, previous, reverse_all, Sequence};

#[derive(Debug, PartialEq, Sequence)]
enum Day { Monday, Tuesday, Wednesday, Thursday, Friday, Saturday, Sunday }

println!("Count: {}", cardinality::<Day>());
println!("Elems: {:?}", all::<Day>().collect::<Vec<_>>());

Count: 7
Elems: [Monday, Tuesday, Wednesday, Thursday, Friday, Saturday, Sunday]


In [17]:
:dep enum-iterator = { version = "2" }

use enum_iterator::{all, Sequence};
use std::fmt;

#[derive(Debug, Sequence)]
enum Direction {
    North,
    South,
    East,
    West,
}

impl fmt::Display for Direction {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        write!(f, "{:?}", self)
    }
}

// Iterate ALL variants automatically
let result: Vec<String> = all::<Direction>().map(|d| d.to_string()).collect();
println!("{:?}", result); // ["North", "South", "East", "West"]

// Or join into a single string
let joined: String = all::<Direction>()
    .map(|d| d.to_string().to_lowercase())
    .collect::<Vec<_>>()
    .join(", ");
println!("{joined}"); // "North, South, East, West"

["North", "South", "East", "West"]
north, south, east, west


In [5]:
// ## Background Processing for the Commmand Line

use std::sync::mpsc;
use std::thread;
use std::time::Duration;

// A simple message type for progress updates
#[derive(Debug)]
enum Update {
    Progress { worker_id: usize, step: u32 },
    Done { worker_id: usize },
}

fn main() {
    let num_workers = 4;
    let (tx, rx) = mpsc::channel::<Update>();

    // Spawn background workers
    let handles: Vec<_> = (0..num_workers)
        .map(|id| {
            let tx = tx.clone();
            thread::spawn(move || {
                for step in 1..=5 {
                    // Simulate work
                    thread::sleep(Duration::from_millis(200 * (id as u64 + 1)));

                    tx.send(Update::Progress { worker_id: id, step })
                        .expect("channel closed unexpectedly");
                }
                tx.send(Update::Done { worker_id: id })
                    .expect("channel closed unexpectedly");
            })
        })
        .collect();

    // Drop the original sender so the channel closes when all workers finish
    drop(tx);

    // Main thread: receive and display summary updates
    for update in rx {
        match update {
            Update::Progress { worker_id, step } => {
                println!("[Worker {worker_id}] Step {step}/5 complete");
            }
            Update::Done { worker_id } => {
                println!("[Worker {worker_id}] ✅ Finished!");
            }
        }
    }

    // Wait for all threads to exit cleanly
    for handle in handles {
        handle.join().expect("worker thread panicked");
    }

    println!("All workers done.");
}